# ЛР № 4. Аналитическая витрина кинофильмов (Movie Industry, 1980–2019)

Исполняемый ноутбук участка `staging → mart` сквозного проекта. Витрина собирается для ответа на **основной аналитический вопрос** из принятого [Project Brief](../docs/project_brief.md) с учётом [решений о качестве ЛР № 3](../docs/data_quality_report.md):

> **Основной вопрос:** Как в период четырёх полных десятилетий с 1980 по 2019 год различаются количество выпущенных фильмов, медианный коэффициент кассовой отдачи (`gross / budget`), доля фильмов с превышением сборов над бюджетом (`gross >= budget`) и медианная оценка IMDb (`score`) в разрезе основных жанров (`genre`), возрастных категорий (`rating`) и десятилетий выпуска?

- **Поддерживаемые будущие метрики:** число фильмов (`count(*)`), медианный коэффициент кассовой отдачи (`median(gross_to_budget_ratio)`), доля условно окупившихся картин (`is_gross_ge_budget`) и медианная оценка IMDb (`median(imdb_score)`).
- **Срезы:** основной жанр (`genre`), прокатный возрастной сертификат (`rating` с флагом шкалы MPAA `nonstandard_rating_flag`) и полное десятилетие выпуска (`release_decade`).
- Описание модели и сопоставление полей приведены в [docs/mart.md](../docs/mart.md).

<a id="input"></a>
## 1. Параметры и входные данные из `staging`

Различаем три временных понятия:
1. **Время события (`release_year`):** календарный год выхода фильма из целевого интервала четырёх полных десятилетий `1980–2019` гг. (`[1980; 2020)`).
2. **Характер источника:** статический снимок публичных карточек IMDb (оценки `score`, голоса `votes` и валовые сборы `gross` отражают накопленные значения на момент формирования набора автором, а не историю их изменений во времени).
3. **Логическая дата загрузки (`source_ingested_on=2026-10-24`):** техническая метка снимка из `staging`, которая переносится в выходной путь `mart` и не подменяет год события.

Ниже задаются пути входа и выхода, загружаются настройки S3 из `.env` и читается `staging.parquet` напрямую из SeaweedFS без повторного запуска ingestion и ЛР № 3.

In [1]:
from datetime import date

DATASET_SLUG = "movie_industry"
SOURCE_INGESTED_ON = "2026-10-24"
MART_NAME = "movies"
STUDY_YEAR_MIN = 1980
STUDY_YEAR_MAX = 2019
MPAA_RATINGS = ('G', 'PG', 'PG-13', 'R', 'NC-17')
EXPECTED_DECADES = ('1980-1989', '1990-1999', '2000-2009', '2010-2019')

date.fromisoformat(SOURCE_INGESTED_ON)

SNAPSHOT_PREFIX = f"{DATASET_SLUG}/source_ingested_on={SOURCE_INGESTED_ON}"
STAGING_KEY = f"{SNAPSHOT_PREFIX}/staging.parquet"
STAGING_URI = f"s3://staging/{STAGING_KEY}"
MART_KEY = f"{SNAPSHOT_PREFIX}/{MART_NAME}.parquet"
MART_URI = f"s3://mart/{MART_KEY}"

print(f"Вход (staging): {STAGING_URI}")
print(f"Выход (mart):   {MART_URI}")

Вход (staging): s3://staging/movie_industry/source_ingested_on=2026-10-24/staging.parquet
Выход (mart):   s3://mart/movie_industry/source_ingested_on=2026-10-24/movies.parquet


In [2]:
import os
import sys
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=False)

required_env = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing_env = [name for name in required_env if not os.getenv(name)]
if missing_env:
    msg = "Не заданы переменные окружения S3: " + ", ".join(missing_env)
    raise RuntimeError(msg)

src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
from analysis.read_raw import configure_s3  # noqa: E402


def sql_literal(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def render_table(headers: list[str], rows: list[tuple[object, ...]]) -> None:
    head_html = "".join(f"<th>{escape(str(col))}</th>" for col in headers)
    body_html = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{head_html}</tr></thead>"
                 f"<tbody>{body_html}</tbody></table>"))


check_results: list[tuple[str, object, object, bool]] = []


def check(label: str, actual: object, expected: object) -> None:
    passed = actual == expected
    check_results.append((label, actual, expected, passed))
    assert passed, f"{label}: получено {actual!r}, ожидалось {expected!r}"


s3_endpoint = os.environ["S3_ENDPOINT"]
s3_endpoint_url = (
    s3_endpoint if s3_endpoint.startswith("http") else f"http://{s3_endpoint}"
)
s3_client = boto3.client(
    "s3",
    endpoint_url=s3_endpoint_url,
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)
staging_before = s3_client.head_object(Bucket="staging", Key=STAGING_KEY)

conn = duckdb.connect()
configure_s3(conn)

In [3]:
conn.execute(f"""
    CREATE OR REPLACE TEMP VIEW staging_movies AS
    SELECT * FROM read_parquet({sql_literal(STAGING_URI)}, hive_partitioning = false)
""")
staging_schema = {
    row[0]: row[1] for row in conn.execute("DESCRIBE staging_movies").fetchall()
}
required_staging_fields = {
    "name",
    "year",
    "genre",
    "rating",
    "score",
    "votes",
    "budget",
    "gross",
    "quality_incomplete_financials",
    "quality_nonstandard_rating",
}
check(
    "Наличие нужных полей в staging",
    required_staging_fields - staging_schema.keys(),
    set(),
)

staging_profile = conn.execute(f"""
    WITH keyed AS (
        SELECT
            *,
            count(*) OVER (PARTITION BY lower(name), year) AS key_cnt
        FROM staging_movies
    )
    SELECT
        count(*) AS total_rows,
        count(*) FILTER (WHERE rating IS NULL) AS null_rating,
        count(*) FILTER (WHERE budget IS NULL) AS null_budget,
        count(*) FILTER (WHERE gross IS NULL) AS null_gross,
        count(*) FILTER (WHERE quality_incomplete_financials) AS fin_flagged,
        count(*) FILTER (WHERE quality_nonstandard_rating) AS rating_flagged,
        count(*) FILTER (
            WHERE name IS NULL OR trim(name) = '' OR year IS NULL OR key_cnt > 1
               OR year < {STUDY_YEAR_MIN} OR year > {STUDY_YEAR_MAX}
               OR score IS NULL OR score < 0 OR score > 10
               OR votes IS NULL OR votes < 1
        ) AS critical_violations
    FROM keyed
""").fetchone()
(
    staging_rows,
    staging_null_rating,
    staging_null_budget,
    staging_null_gross,
    staging_fin_flagged,
    staging_rating_flagged,
    staging_critical,
) = staging_profile
check("Критические нарушения ЛР № 3 на входе", staging_critical, 0)

render_table(
    ["Строк staging", "NULL rating", "NULL budget", "NULL gross",
     "Флаг финансов", "Флаг рейтинга", "Крит. нарушений"],
    [staging_profile],
)
render_table(
    ["Используемое поле staging", "Тип в staging"],
    [(col, staging_schema[col]) for col in sorted(required_staging_fields)],
)

Строк staging,NULL rating,NULL budget,NULL gross,Флаг финансов,Флаг рейтинга,Крит. нарушений
7640,68,2159,177,2213,419,0


Используемое поле staging,Тип в staging
budget,DOUBLE
genre,VARCHAR
gross,DOUBLE
name,VARCHAR
quality_incomplete_financials,BOOLEAN
quality_nonstandard_rating,BOOLEAN
rating,VARCHAR
score,DOUBLE
votes,BIGINT
year,INTEGER


<a id="model"></a>
## 2. Гранулярность, ключ, меры и структура витрины

> **Одна строка таблицы `movies` представляет один полнометражный кинофильм, вышедший в период четырёх полных десятилетий 1980–2019 гг., из подготовленного снимка `staging`.**

**Идентификация строки:** в источнике нет суррогатного числового ID, а одно только название повторяется у ремейков и одноимённых картин разных лет (`157` повторов `lower(name)` в `staging`), но составной естественный ключ `(lower(movie_title), release_year)` однозначно определяет каждый фильм (в ЛР № 3 подтверждено отсутствие пропусков и повторов; ниже это повторно проверяется на сохранённой витрине). Искусственный технический ключ и `DISTINCT` не применяются.

**Выбор структуры:** используется **одна широкая таблица `movies`** без отдельных таблиц измерений и без `JOIN`. Все срезы основного вопроса (`genre`, `rating`, `release_decade`) уже определены на уровне карточки фильма, имеют малую мощность (`19` жанров, `5` категорий MPAA, `4` десятилетия) и не содержат внешних атрибутов справочников.

| Мера / будущий показатель | Единица измерения | Использование в основном вопросе | Правило агрегации и объединения срезов |
|---|---|---|---|
| Число фильмов (`count(*)`) | шт. (фильмы) | Объём выпуска по жанрам, рейтингам и десятилетиям (включая помеченные строки) | **Аддитивно** для непересекающихся групп одного снимка |
| `budget_usd`, `gross_usd` | номинальные USD | Базовые величины для проверки и расчёта кассовой отдачи (только `NOT incomplete_financials_flag`) | Допускают суммирование внутри непересекающихся групп, но без поправки на инфляцию и не равны всему рынку |
| `gross_to_budget_ratio` (`gross / budget`) | отношение (раз) | Расчёт `median(gross_to_budget_ratio)` на подмножестве полных финансов | **Неаддитивно:** отношения и групповые медианы нельзя складывать или усреднять — пересчитывать из строк среза |
| `is_gross_ge_budget` (`gross >= budget`) | булев признак (для доли в `%`) | Доля фильмов с условным покрытием бюджета валовой кассой среди `NOT incomplete_financials_flag` | Доли групп не складываются; при объединении групп суммируются отдельно числитель и знаменатель |
| `imdb_score` (при контроле `imdb_votes`) | баллы IMDb (`[0; 10]`) | Расчёт `median(imdb_score)` по всем строкам среза | **Неаддитивно:** баллы и медианы групп не складываются — медиана пересчитывается по строкам среза |

**Сохранение решений ЛР № 3:** все `7 640` строк `staging` переносятся в витрину без фильтрации. Для `2 213` фильмов с `incomplete_financials_flag = true` производные поля `gross_to_budget_ratio` и `is_gross_ge_budget` принимают значение `NULL` (окупаемость неизвестна и не подменяется нулём или `false`), при этом исходные частично заполненные `budget_usd` / `gross_usd` и нестандартные значения `rating` (`419` строк с `nonstandard_rating_flag = true`) сохраняются без изменений.

In [4]:
staging_ranges = conn.execute("""
    SELECT
        min(year), max(year),
        min(score), max(score),
        min(votes), max(votes),
        min(budget), max(budget),
        min(gross), max(gross),
        count(*) - count(DISTINCT lower(name)) AS title_only_repeats,
        count(*) FILTER (WHERE NOT quality_incomplete_financials) AS complete_fin_rows,
        count(*) FILTER (
            WHERE NOT quality_incomplete_financials AND gross >= budget
        ) AS recoup_rows
    FROM staging_movies
""").fetchone()
render_table(
    ["Мин. год", "Макс. год", "Мин. score", "Макс. score",
     "Повторов только name", "Полных фин. пар", "Из них gross >= budget"],
    [(
        staging_ranges[0],
        staging_ranges[1],
        staging_ranges[2],
        staging_ranges[3],
        staging_ranges[10],
        staging_ranges[11],
        staging_ranges[12],
    )],
)

Мин. год,Макс. год,Мин. score,Макс. score,Повторов только name,Полных фин. пар,Из них gross >= budget
1980,2019,1.9,9.3,157,5427,3677


<a id="transform"></a>
## 3. SQL-преобразование и сборка витрины `movies`

Явно отбираем 13 полей, необходимых для основного вопроса `Project Brief`:
- составной ключ и временные атрибуты: `movie_title`, `release_year`, `release_decade` (`'1980-1989'`, `'1990-1999'`, `'2000-2009'`, `'2010-2019'`);
- категориальные срезы: `genre`, `rating`;
- зрительские и финансовые меры: `imdb_score`, `imdb_votes`, `budget_usd`, `gross_usd`, а также вычисляемые на полных финансовых парах `gross_to_budget_ratio` и `is_gross_ge_budget`;
- перенесённые флаги качества ЛР № 3: `incomplete_financials_flag`, `nonstandard_rating_flag`.

Детерминированная сортировка `ORDER BY s.year, lower(s.name)` сохраняет все строки и обеспечивает побайтовую воспроизводимость Parquet-файла.

In [5]:
conn.execute("""
    CREATE OR REPLACE TEMP TABLE movies_prepared AS
    SELECT
        s.name AS movie_title,
        s.year AS release_year,
        concat((s.year // 10) * 10, '-', (s.year // 10) * 10 + 9) AS release_decade,
        s.genre AS genre,
        s.rating AS rating,
        s.score AS imdb_score,
        s.votes AS imdb_votes,
        s.budget AS budget_usd,
        s.gross AS gross_usd,
        CASE
            WHEN NOT s.quality_incomplete_financials THEN s.gross / s.budget
            ELSE NULL
        END AS gross_to_budget_ratio,
        CASE
            WHEN NOT s.quality_incomplete_financials THEN s.gross >= s.budget
            ELSE NULL
        END AS is_gross_ge_budget,
        s.quality_incomplete_financials AS incomplete_financials_flag,
        s.quality_nonstandard_rating AS nonstandard_rating_flag
    FROM staging_movies AS s
    ORDER BY s.year, lower(s.name)
""")
prepared_rows = conn.execute("SELECT count(*) FROM movies_prepared").fetchone()[0]
check("Сохранение всех строк при сборке витрины", prepared_rows, staging_rows)

render_table(
    ["movie_title", "release_year", "release_decade", "genre", "rating",
     "imdb_score", "gross_to_budget_ratio", "is_gross_ge_budget",
     "incomplete_financials_flag", "nonstandard_rating_flag"],
    conn.execute("""
        SELECT
            movie_title,
            release_year,
            release_decade,
            genre,
            rating,
            imdb_score,
            round(gross_to_budget_ratio, 3),
            is_gross_ge_budget,
            incomplete_financials_flag,
            nonstandard_rating_flag
        FROM movies_prepared
        ORDER BY release_year, lower(movie_title)
        LIMIT 5
    """).fetchall(),
)

movie_title,release_year,release_decade,genre,rating,imdb_score,gross_to_budget_ratio,is_gross_ge_budget,incomplete_financials_flag,nonstandard_rating_flag
9 to 5,1980,1980-1989,Comedy,PG,6.9,10.33,True,False,False
Airplane!,1980,1980-1989,Comedy,PG,7.7,23.844,True,False,False
Altered States,1980,1980-1989,Horror,R,6.9,None,None,True,False
American Gigolo,1980,1980-1989,Crime,R,6.2,4.738,True,False,False
Any Which Way You Can,1980,1980-1989,Action,PG,6.1,4.712,True,False,False


<a id="write"></a>
## 4. Запись витрины в бакет `mart` и повторное чтение из S3

Заменяем целевой объект `movies.parquet` идемпотентно: удаляем предыдущую версию по ключу и материализуем таблицу в Parquet (`COMPRESSION ZSTD`). Затем открываем **отдельное соединение DuckDB** и читаем записанный файл напрямую из `s3://mart/...` с `hive_partitioning = false`, чтобы проверить реальную физическую схему сохранённого объекта.

In [6]:
assert MART_URI == f"s3://mart/{SNAPSHOT_PREFIX}/movies.parquet"
assert MART_KEY == f"{SNAPSHOT_PREFIX}/movies.parquet"

s3_client.head_bucket(Bucket="mart")
s3_client.delete_object(Bucket="mart", Key=MART_KEY)
conn.execute(f"""
    COPY movies_prepared TO {sql_literal(MART_URI)}
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

published_conn = duckdb.connect()
configure_s3(published_conn)
published_conn.execute(f"""
    CREATE OR REPLACE TEMP VIEW published_movies AS
    SELECT * FROM read_parquet({sql_literal(MART_URI)}, hive_partitioning = false)
""")
mart_head = s3_client.head_object(Bucket="mart", Key=MART_KEY)
render_table(
    ["Опубликованный S3-объект", "Размер Parquet, байт"],
    [(MART_URI, mart_head["ContentLength"])],
)

Опубликованный S3-объект,"Размер Parquet, байт"
s3://mart/movie_industry/source_ingested_on=2026-10-24/movies.parquet,191863


<a id="checks"></a>
## 5. Проверки сохранённой витрины

Все проверки ниже выполняются исключительно по прочитанному из S3 представлению `published_movies` в независимом соединении DuckDB.

### 5.1. Количество строк и физическая схема

In [7]:
mart_rows = published_conn.execute(
    "SELECT count(*) FROM published_movies"
).fetchone()[0]
excluded_rows = staging_rows - prepared_rows
check("Число исключённых строк", excluded_rows, 0)
check("Баланс строк staging = mart + исключено", staging_rows, mart_rows + excluded_rows)
check("Число строк после чтения из S3", mart_rows, prepared_rows)

expected_schema = [
    ("movie_title", "VARCHAR"),
    ("release_year", "INTEGER"),
    ("release_decade", "VARCHAR"),
    ("genre", "VARCHAR"),
    ("rating", "VARCHAR"),
    ("imdb_score", "DOUBLE"),
    ("imdb_votes", "BIGINT"),
    ("budget_usd", "DOUBLE"),
    ("gross_usd", "DOUBLE"),
    ("gross_to_budget_ratio", "DOUBLE"),
    ("is_gross_ge_budget", "BOOLEAN"),
    ("incomplete_financials_flag", "BOOLEAN"),
    ("nonstandard_rating_flag", "BOOLEAN"),
]
actual_schema = [
    (row[0], row[1])
    for row in published_conn.execute("DESCRIBE published_movies").fetchall()
]
check("Имена, порядок и типы 13 полей витрины", actual_schema, expected_schema)

render_table(
    ["Вход (staging)", "Подготовлено", "Прочитано из mart", "Исключено"],
    [(staging_rows, prepared_rows, mart_rows, excluded_rows)],
)
render_table(["Поле витрины", "Тип DuckDB"], actual_schema)

Вход (staging),Подготовлено,Прочитано из mart,Исключено
7640,7640,7640,0


Поле витрины,Тип DuckDB
movie_title,VARCHAR
release_year,INTEGER
release_decade,VARCHAR
genre,VARCHAR
rating,VARCHAR
imdb_score,DOUBLE
imdb_votes,BIGINT
budget_usd,DOUBLE
gross_usd,DOUBLE
gross_to_budget_ratio,DOUBLE


### 5.2. Составной бизнес-ключ и обязательные поля

Проверяем отсутствие `NULL` и пустых строк в каждом компоненте составного ключа `(movie_title, release_year)` и отсутствие повторов как при точном, так и при регистронезависимом сравнении `(lower(movie_title), release_year)`. Сверяем также отсутствие `NULL` во всех обязательных полях и ожидаемое число допустимых пропусков в `rating`, `budget_usd`, `gross_usd` и производных финансовых показателях.

In [8]:
key_checks = published_conn.execute("""
    SELECT
        count(*) FILTER (
            WHERE movie_title IS NULL OR trim(movie_title) = ''
        ) AS null_or_empty_title,
        count(*) FILTER (WHERE release_year IS NULL) AS null_year,
        (SELECT count(*) FROM (
            SELECT movie_title, release_year
            FROM published_movies
            WHERE movie_title IS NOT NULL AND release_year IS NOT NULL
            GROUP BY movie_title, release_year
            HAVING count(*) > 1
        )) AS exact_key_dups,
        (SELECT count(*) FROM (
            SELECT lower(movie_title), release_year
            FROM published_movies
            WHERE movie_title IS NOT NULL AND release_year IS NOT NULL
            GROUP BY lower(movie_title), release_year
            HAVING count(*) > 1
        )) AS lower_key_dups
    FROM published_movies
""").fetchone()
check("Пропуски или пустые строки movie_title", key_checks[0], 0)
check("Пропуски release_year", key_checks[1], 0)
check("Повторы ключа (movie_title, release_year)", key_checks[2], 0)
check("Повторы ключа (lower(movie_title), release_year)", key_checks[3], 0)
render_table(
    ["Пустой movie_title", "NULL release_year",
     "Дубликаты точного ключа", "Дубликаты lower-ключа"],
    [key_checks],
)

mandatory_cols = (
    "movie_title",
    "release_year",
    "release_decade",
    "genre",
    "imdb_score",
    "imdb_votes",
    "incomplete_financials_flag",
    "nonstandard_rating_flag",
)
null_summary_rows = []
for col in mandatory_cols:
    null_cnt = published_conn.execute(
        f"SELECT count(*) FROM published_movies WHERE {col} IS NULL"
    ).fetchone()[0]
    check(f"NULL в обязательном поле {col}", null_cnt, 0)
    null_summary_rows.append((col, null_cnt, 0))

nullable_expectations = (
    ("rating", staging_null_rating),
    ("budget_usd", staging_null_budget),
    ("gross_usd", staging_null_gross),
    ("gross_to_budget_ratio", staging_fin_flagged),
    ("is_gross_ge_budget", staging_fin_flagged),
)
for col, expected_nulls in nullable_expectations:
    null_cnt = published_conn.execute(
        f"SELECT count(*) FROM published_movies WHERE {col} IS NULL"
    ).fetchone()[0]
    check(f"Ожидаемое число NULL в поле {col}", null_cnt, expected_nulls)
    null_summary_rows.append((col, null_cnt, expected_nulls))

render_table(["Поле витрины", "Фактически NULL", "Ожидалось NULL"], null_summary_rows)

Пустой movie_title,NULL release_year,Дубликаты точного ключа,Дубликаты lower-ключа
0,0,0,0


Поле витрины,Фактически NULL,Ожидалось NULL
movie_title,0,0
release_year,0,0
release_decade,0,0
genre,0,0
imdb_score,0,0
imdb_votes,0,0
incomplete_financials_flag,0,0
nonstandard_rating_flag,0,0
rating,68,68
budget_usd,2159,2159


### 5.3. Меры и вычисленные поля

Проверяем соответствие диапазонов мер правилам ЛР № 3 (`1980 <= release_year <= 2019`, `0 <= imdb_score <= 10`, `imdb_votes >= 1`, положительность и конечность заполненных `budget_usd`, `gross_usd` и `gross_to_budget_ratio`), совпадение минимумов и максимумов со `staging`, а также точное выполнение формул `release_decade`, `gross_to_budget_ratio` и `is_gross_ge_budget`.

In [9]:
mart_ranges = published_conn.execute("""
    SELECT
        min(release_year), max(release_year),
        min(imdb_score), max(imdb_score),
        min(imdb_votes), max(imdb_votes),
        min(budget_usd), max(budget_usd),
        min(gross_usd), max(gross_usd),
        min(gross_to_budget_ratio), max(gross_to_budget_ratio),
        count(*) FILTER (WHERE is_gross_ge_budget) AS recoup_true,
        count(*) FILTER (WHERE is_gross_ge_budget = false) AS recoup_false
    FROM published_movies
""").fetchone()
check("Диапазон release_year сохранён", mart_ranges[0:2], staging_ranges[0:2])
check("Диапазон imdb_score сохранён", mart_ranges[2:4], staging_ranges[2:4])
check("Диапазон imdb_votes сохранён", mart_ranges[4:6], staging_ranges[4:6])
check("Диапазон budget_usd сохранён", mart_ranges[6:8], staging_ranges[6:8])
check("Диапазон gross_usd сохранён", mart_ranges[8:10], staging_ranges[8:10])
check("Число фильмов с gross >= budget", mart_ranges[12], staging_ranges[12])
check(
    "Сумма true и false в is_gross_ge_budget равна числу полных фин. пар",
    mart_ranges[12] + mart_ranges[13],
    staging_ranges[11],
)

distinct_decades = tuple(
    row[0]
    for row in published_conn.execute(
        "SELECT DISTINCT release_decade FROM published_movies ORDER BY 1"
    ).fetchall()
)
check("Набор значений release_decade", distinct_decades, EXPECTED_DECADES)

derived_errors = published_conn.execute(f"""
    SELECT count(*)
    FROM published_movies
    WHERE release_year NOT BETWEEN {STUDY_YEAR_MIN} AND {STUDY_YEAR_MAX}
       OR release_decade IS DISTINCT FROM
          concat((release_year // 10) * 10, '-', (release_year // 10) * 10 + 9)
       OR imdb_score < 0 OR imdb_score > 10 OR NOT isfinite(imdb_score)
       OR imdb_votes < 1
       OR (budget_usd IS NOT NULL AND (budget_usd <= 0 OR NOT isfinite(budget_usd)))
       OR (gross_usd IS NOT NULL AND (gross_usd <= 0 OR NOT isfinite(gross_usd)))
       OR (gross_to_budget_ratio IS NOT NULL
           AND (gross_to_budget_ratio <= 0 OR NOT isfinite(gross_to_budget_ratio)))
       OR gross_to_budget_ratio IS DISTINCT FROM (
           CASE WHEN NOT incomplete_financials_flag THEN gross_usd / budget_usd END
       )
       OR is_gross_ge_budget IS DISTINCT FROM (
           CASE WHEN NOT incomplete_financials_flag THEN gross_usd >= budget_usd END
       )
""").fetchone()[0]
check("Нарушения диапазонов и формул вычисленных полей", derived_errors, 0)

render_table(
    ["Показатель", "Минимум", "Максимум", "Единица"],
    [
        ("release_year", mart_ranges[0], mart_ranges[1], "год"),
        ("imdb_score", mart_ranges[2], mart_ranges[3], "баллы IMDb"),
        ("imdb_votes", mart_ranges[4], mart_ranges[5], "голоса"),
        ("budget_usd", mart_ranges[6], mart_ranges[7], "USD (номинал)"),
        ("gross_usd", mart_ranges[8], mart_ranges[9], "USD (номинал)"),
        (
            "gross_to_budget_ratio",
            round(mart_ranges[10], 6),
            round(mart_ranges[11], 2),
            "отношение (раз)",
        ),
    ],
)

Показатель,Минимум,Максимум,Единица
release_year,1980,2019,год
imdb_score,1.9,9.3,баллы IMDb
imdb_votes,34,2400000,голоса
budget_usd,3000.0,356000000.0,USD (номинал)
gross_usd,309.0,2847246203.0,USD (номинал)
gross_to_budget_ratio,2.1e-05,12890.39,отношение (раз)


### 5.4. Флаги качества и точный перенос значений (`EXCEPT ALL`)

Проверяем, что оба флага ЛР № 3 перенесены в точном соответствии со `staging`, а исходные значения в помеченных строках (например, известные `gross_usd` при пропущенном `budget_usd` или текстовые метки `'Not Rated'`, `'Unrated'`, `'TV-MA'` в поле `rating`) сохранены без замены на `NULL` или ноль. Двусторонняя сверка `EXCEPT ALL` подтверждает побайтовое совпадение всех 13 полей и кратности строк.

In [10]:
flag_checks = published_conn.execute(f"""
    SELECT
        count(*) FILTER (WHERE incomplete_financials_flag) AS fin_flagged,
        count(*) FILTER (
            WHERE incomplete_financials_flag IS DISTINCT FROM
                  (budget_usd IS NULL OR budget_usd <= 0
                   OR gross_usd IS NULL OR gross_usd <= 0)
        ) AS fin_flag_mismatch,
        count(*) FILTER (
            WHERE incomplete_financials_flag AND gross_usd IS NOT NULL
        ) AS partial_gross_preserved,
        count(*) FILTER (WHERE nonstandard_rating_flag) AS rating_flagged,
        count(*) FILTER (
            WHERE nonstandard_rating_flag IS DISTINCT FROM
                  (rating IS NULL OR rating NOT IN {MPAA_RATINGS})
        ) AS rating_flag_mismatch,
        count(*) FILTER (
            WHERE nonstandard_rating_flag AND rating IS NOT NULL
        ) AS nonmpaa_labels_preserved
    FROM published_movies
""").fetchone()

staging_partial = conn.execute("""
    SELECT
        count(*) FILTER (
            WHERE quality_incomplete_financials AND gross IS NOT NULL
        ),
        count(*) FILTER (
            WHERE quality_nonstandard_rating AND rating IS NOT NULL
        )
    FROM staging_movies
""").fetchone()

check("Число строк с incomplete_financials_flag", flag_checks[0], staging_fin_flagged)
check("Несоответствия incomplete_financials_flag", flag_checks[1], 0)
check(
    "Сохранены известные gross_usd в строках с флагом финансов",
    flag_checks[2],
    staging_partial[0],
)
check("Число строк с nonstandard_rating_flag", flag_checks[3], staging_rating_flagged)
check("Несоответствия nonstandard_rating_flag", flag_checks[4], 0)
check(
    "Сохранены непустые метки rating вне шкалы MPAA",
    flag_checks[5],
    staging_partial[1],
)
render_table(
    ["Флаг финансов", "Расхождений фин.", "Сохранено частичных gross",
     "Флаг рейтинга", "Расхождений рейт.", "Сохранено не-MPAA меток"],
    [flag_checks],
)

published_conn.execute(f"""
    CREATE OR REPLACE TEMP VIEW expected_movies AS
    SELECT
        s.name AS movie_title,
        s.year AS release_year,
        concat((s.year // 10) * 10, '-', (s.year // 10) * 10 + 9) AS release_decade,
        s.genre AS genre,
        s.rating AS rating,
        s.score AS imdb_score,
        s.votes AS imdb_votes,
        s.budget AS budget_usd,
        s.gross AS gross_usd,
        CASE
            WHEN NOT s.quality_incomplete_financials THEN s.gross / s.budget
            ELSE NULL
        END AS gross_to_budget_ratio,
        CASE
            WHEN NOT s.quality_incomplete_financials THEN s.gross >= s.budget
            ELSE NULL
        END AS is_gross_ge_budget,
        s.quality_incomplete_financials AS incomplete_financials_flag,
        s.quality_nonstandard_rating AS nonstandard_rating_flag
    FROM read_parquet({sql_literal(STAGING_URI)}, hive_partitioning = false) AS s
""")
diff_rows = published_conn.execute("""
    SELECT 'Отсутствуют в mart' AS direction, count(*) AS diff_cnt
    FROM (SELECT * FROM expected_movies EXCEPT ALL SELECT * FROM published_movies)
    UNION ALL
    SELECT 'Лишние в mart' AS direction, count(*) AS diff_cnt
    FROM (SELECT * FROM published_movies EXCEPT ALL SELECT * FROM expected_movies)
""").fetchall()
for direction, diff_cnt in diff_rows:
    check(direction, diff_cnt, 0)
render_table(["Направление сверки EXCEPT ALL", "Различающихся строк"], diff_rows)

Флаг финансов,Расхождений фин.,Сохранено частичных gross,Флаг рейтинга,Расхождений рейт.,Сохранено не-MPAA меток
2213,0,2036,419,0,351


Направление сверки EXCEPT ALL,Различающихся строк
Отсутствуют в mart,0
Лишние в mart,0


### 5.5. Пригодность витрины для выбранного вопроса

Контрольный запрос группирует сохранённую витрину по десятилетиям и жанрам (`release_decade`, `genre`), подсчитывая общее число фильмов, число картин с полной финансовой парой для расчёта `gross / budget` и число картин со стандартным рейтингом MPAA. Сумма записей по всем группам совпадает с общим размером витрины (`7 640`), подтверждая готовность срезов и флагов к расчёту метрик в следующей лабораторной работе.

In [11]:
decade_genre_groups = published_conn.execute("""
    SELECT
        release_decade,
        genre,
        count(*) AS films_count,
        count(*) FILTER (
            WHERE NOT incomplete_financials_flag
        ) AS complete_fin_films,
        count(*) FILTER (
            WHERE NOT nonstandard_rating_flag
        ) AS mpaa_films
    FROM published_movies
    GROUP BY release_decade, genre
    ORDER BY films_count DESC, release_decade, genre
""").fetchall()
total_grouped_films = sum(row[2] for row in decade_genre_groups)
total_grouped_fin = sum(row[3] for row in decade_genre_groups)
check("Баланс контрольной группировки по всем фильмам", total_grouped_films, mart_rows)
check(
    "Баланс контрольной группировки по полным фин. парам",
    total_grouped_fin,
    staging_ranges[11],
)
render_table(
    ["Десятилетие", "Жанр", "Всего фильмов",
     "С полной фин. парой", "Со шкалой MPAA"],
    decade_genre_groups[:10],
)
render_table(
    ["Всего комбинаций (десятилетие × жанр)", "Сумма фильмов", "Сумма полных фин. пар"],
    [(len(decade_genre_groups), total_grouped_films, total_grouped_fin)],
)

Десятилетие,Жанр,Всего фильмов,С полной фин. парой,Со шкалой MPAA
1990-1999,Comedy,678,436,656
2000-2009,Comedy,576,465,559
1980-1989,Comedy,565,278,555
2010-2019,Action,546,472,508
2000-2009,Action,446,398,434
2010-2019,Comedy,423,317,399
1990-1999,Drama,415,232,374
2010-2019,Drama,403,238,335
1990-1999,Action,385,319,382
2000-2009,Drama,380,273,333


Всего комбинаций (десятилетие × жанр),Сумма фильмов,Сумма полных фин. пар
62,7640,5427


### 5.6. Контрольная сводка и неизменность входного `staging`

Убеждаемся, что входной объект `staging.parquet` не изменился за время сборки витрины (`ETag`, размер и время модификации совпадают), и выводим итоговый протокол всех автоматических проверок.

In [12]:
staging_after = s3_client.head_object(Bucket="staging", Key=STAGING_KEY)
check(
    "Входной объект staging.parquet не изменён",
    (
        staging_after["ContentLength"],
        staging_after["ETag"],
        staging_after["LastModified"] == staging_before["LastModified"],
    ),
    (staging_before["ContentLength"], staging_before["ETag"], True),
)

passed_checks = sum(1 for _, _, _, ok in check_results if ok)
render_table(
    ["Проверка", "Получено", "Ожидалось", "Статус"],
    [
        (label, actual, expected, "Пройдена" if ok else "Ошибка")
        for label, actual, expected, ok in check_results
    ],
)
render_table(
    ["Всего проверок", "Пройдено", "Строк в mart",
     "Флаг финансов", "Флаг рейтинга"],
    [(
        len(check_results),
        passed_checks,
        mart_rows,
        flag_checks[0],
        flag_checks[3],
    )],
)

published_conn.close()
conn.close()

Проверка,Получено,Ожидалось,Статус
Наличие нужных полей в staging,set(),set(),Пройдена
Критические нарушения ЛР № 3 на входе,0,0,Пройдена
Сохранение всех строк при сборке витрины,7640,7640,Пройдена
Число исключённых строк,0,0,Пройдена
Баланс строк staging = mart + исключено,7640,7640,Пройдена
Число строк после чтения из S3,7640,7640,Пройдена
"Имена, порядок и типы 13 полей витрины","[('movie_title', 'VARCHAR'), ('release_year', 'INTEGER'), ('release_decade', 'VARCHAR'), ('genre', 'VARCHAR'), ('rating', 'VARCHAR'), ('imdb_score', 'DOUBLE'), ('imdb_votes', 'BIGINT'), ('budget_usd', 'DOUBLE'), ('gross_usd', 'DOUBLE'), ('gross_to_budget_ratio', 'DOUBLE'), ('is_gross_ge_budget', 'BOOLEAN'), ('incomplete_financials_flag', 'BOOLEAN'), ('nonstandard_rating_flag', 'BOOLEAN')]","[('movie_title', 'VARCHAR'), ('release_year', 'INTEGER'), ('release_decade', 'VARCHAR'), ('genre', 'VARCHAR'), ('rating', 'VARCHAR'), ('imdb_score', 'DOUBLE'), ('imdb_votes', 'BIGINT'), ('budget_usd', 'DOUBLE'), ('gross_usd', 'DOUBLE'), ('gross_to_budget_ratio', 'DOUBLE'), ('is_gross_ge_budget', 'BOOLEAN'), ('incomplete_financials_flag', 'BOOLEAN'), ('nonstandard_rating_flag', 'BOOLEAN')]",Пройдена
Пропуски или пустые строки movie_title,0,0,Пройдена
Пропуски release_year,0,0,Пройдена
"Повторы ключа (movie_title, release_year)",0,0,Пройдена


Всего проверок,Пройдено,Строк в mart,Флаг финансов,Флаг рейтинга
44,44,7640,2213,419


<a id="result"></a>
## 6. Итог сборки витрины

Описание модели, таблица происхождения всех 13 полей и перечень ограничений оформлены в [docs/mart.md](../docs/mart.md). Детальные результаты проверок сохранены выше в [разделе 5](#checks).

- **Выходной объект в SeaweedFS:**
  ```text
  s3://mart/movie_industry/source_ingested_on=2026-10-24/movies.parquet
  ```
- **Баланс строк:** `7 640 входных строк = 7 640 сохранённых строк + 0 исключённых строк`. Гранулярность (один фильм `1980–2019` гг. на строку) и уникальность составного ключа `(lower(movie_title), release_year)` полностью сохранены.
- **Решения о качестве и ограничения:** `2 213` фильмов с неполной финансовой парой (`incomplete_financials_flag = true`) и `419` фильмов с пустым или внерейтинговым сертификатом (`nonstandard_rating_flag = true`) перенесены в витрину без искажения исходных значений. Выборка для расчёта `gross / budget` составляет `5 427` фильмов (`3 677` с `gross_usd >= budget_usd`), денежные суммы выражены в номинальных USD без поправки на инфляцию, а условие `gross_usd >= budget_usd` отражает условное покрытие производственного бюджета валовой кассой, а не чистую рентабельность киностудии.